# 08 — Stage-4 replication-failure report

Stage 2 failed its capability and known-narration positive-control gates, so
the three Stage-3 notebooks executed only prerequisite guards. This notebook
is intentionally model-free. It validates those persisted skips, records the
requested legacy predictor comparison with provenance, and closes the run
without making a hypothesis-level inference.

In [1]:
import json
import os
import sys
from pathlib import Path

ROOT = Path('/home/jovyan/j-space-thoughts')
os.chdir(ROOT)
sys.path.insert(0, str(ROOT))

metrics = json.loads((ROOT / 'results/metrics.json').read_text())
repair = metrics['repair_v2']
stage2 = repair['stage2_recalibration']
ledger = repair['gate_ledger']
assert stage2['status'] == 'FAIL'
assert stage2['stage4_required'] is True
assert stage2['stage3_allowed'] is False
assert ledger['stage3_science'] == 'SKIPPED_PREREQUISITE'
for number in ('05', '06', '07'):
    row = repair['stage3_notebooks'][number]
    assert row['status'] == 'SKIPPED_PREREQUISITE'
    assert row['science_executed'] is False
    assert row['model_inference_run'] is False
print({
    'stage2': stage2['status'],
    'blocking_criteria': [
        name for name, passed in stage2['criteria'].items() if not passed
    ],
    'stage3_science': ledger['stage3_science'],
    'model_loaded': False,
})

{'stage2': 'FAIL', 'blocking_criteria': ['capability_preserved', 'g_pos_reproduced'], 'stage3_science': 'SKIPPED_PREREQUISITE', 'model_loaded': False}


In [2]:
from src.v2_final_report import persist_stage4

metrics = persist_stage4()
stage4 = metrics['repair_v2']['stage4_report']
print(json.dumps({
    'classification': stage4['classification'],
    'custom_swap': stage4['custom_swap'],
    'calibration_blockers': stage4['calibration_blockers'],
    'predictions': stage4['predictions'],
    'skipped_notebooks': stage4['skipped_notebooks'],
    'legacy_fallback_comparison': stage4['legacy_fallback_comparison'],
    'claim_boundary': stage4['claim_boundary'],
}, indent=2))

{
  "classification": "OPEN_MODEL_INSTRUMENT_REPLICATION_FAILURE",
  "custom_swap": {
    "status": "PASS",
    "n_pass": 3,
    "n_required": 3,
    "stage2_reverification": "PASS",
    "configuration": "exact-label raw J-Lens directions for G-SWAP; leading-label raw directions for language controls; alpha=2; original prompt positions"
  },
  "calibration_blockers": [
    {
      "gate": "capability preservation",
      "key": "capability",
      "role": "unrelated-text delta NLL",
      "status": "FAIL",
      "evidence": {
        "mean_delta_nll": 0.6233231921990713,
        "mean_abs_delta_nll": 0.6690807243188223,
        "threshold": 0.25,
        "criterion": "absolute grand mean and every intervention-bank mean delta NLL < 0.25"
      }
    },
    {
      "gate": "G-POS",
      "key": "g_pos",
      "role": "known-narration positive control",
      "status": "FAIL",
      "evidence": {
        "n_reproduced": 1,
        "n_passages": 8,
        "categories_reproduced": [
     

In [3]:
report = (ROOT / 'results/RESULTS.md').read_text()
ledger = metrics['repair_v2']['gate_ledger']
assert ledger['stage4_report'] == 'COMPLETE'
assert ledger['stage3_science'] == 'SKIPPED_PREREQUISITE'
assert report.count('## Stage 4 — replication-failure fallback') == 1
assert 'P1 | **NOT_TESTED**' in report
assert 'P2 | **NOT_TESTED**' in report
assert 'P3 | **NOT_TESTED**' in report
assert 'does not establish that the WRITE-versus-READ hypothesis is' in report
assert all((ROOT / 'results' / row['path']).is_file() for row in stage4['valid_figures'])
print({
    'stage4_report': ledger['stage4_report'],
    'valid_figure_ids': [row['id'] for row in stage4['valid_figures']],
    'final_conclusion': metrics['repair_v2']['current_allowed_conclusion'],
})

{'stage4_report': 'COMPLETE', 'valid_figure_ids': ['F0', 'G-SWAP', 'G-DIR', 'F5', 'F3'], 'final_conclusion': 'STAGE4_REPLICATION_FAILURE_NO_HYPOTHESIS_INFERENCE'}
